# 04 — Model Comparison & Submission

Compares Logistic Regression, Random Forest, XGBoost, and LightGBM using
5-fold stratified cross-validation (Macro F1), tunes the decision threshold,
and writes the best submission CSV.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.model_selection import StratifiedKFold, cross_val_score

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))
sns.set_theme(style='whitegrid')

from src.features import (
    ID_COL, TARGET_COL,
    build_preprocessor, load_train_test,
    prepare_features, split_features_target,
)
from src.train import _make_clf, _best_threshold, _get_col_groups

train_raw, test_raw = load_train_test(ROOT / 'data')
X_raw, y = split_features_target(train_raw)
test_ids = test_raw[ID_COL]

X_train = prepare_features(X_raw)
X_test  = prepare_features(test_raw)
print('Train:', X_train.shape, '  Test:', X_test.shape)

## 1. Build pipelines for each model

Tree-based models use no `StandardScaler` (it doesn't affect splits).
Logistic Regression keeps the scaler.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

num_cols, cat_cols = _get_col_groups(X_train)

def make_pipeline(model_name: str) -> Pipeline:
    clf = _make_clf(model_name)
    needs_scaling = (model_name == 'lr')

    num_steps = [('imputer', SimpleImputer(strategy='median'))]
    if needs_scaling:
        num_steps.append(('scaler', StandardScaler()))

    cat_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False, max_categories=50)),
    ])

    prep = ColumnTransformer([
        ('num', Pipeline(num_steps), num_cols),
        ('cat', cat_pipe, cat_cols),
    ])
    return Pipeline([('prep', prep), ('clf', clf)])

MODELS = ['lr', 'rf', 'xgb', 'lgbm']
print('Models to compare:', MODELS)

## 2. Cross-validation — Macro F1

5-fold stratified CV keeps class proportions consistent across folds.
This takes ~5-10 min for RF/XGBoost/LightGBM.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = {}

for name in MODELS:
    pipe = make_pipeline(name)
    scores = cross_val_score(pipe, X_train, y, cv=cv, scoring='f1_macro', n_jobs=-1)
    results[name] = scores
    print(f'{name:6s}  CV f1_macro: {scores.mean():.4f} ± {scores.std():.4f}')

results_df = pd.DataFrame(results)
results_df

In [ ]:
ax = results_df.boxplot(figsize=(8, 4))
ax.set_title('5-fold CV Macro F1 by model')
ax.set_ylabel('Macro F1')
ax.axhline(results_df['lr'].mean(), color='grey', linestyle='--', label='LR baseline')
ax.legend()
plt.tight_layout(); plt.show()

## 3. Decision threshold tuning

The default threshold (0.5) is not optimal for Macro F1 on imbalanced data.
We find the threshold that maximises Macro F1 on a held-out validation split.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y, test_size=0.2, stratify=y, random_state=42
)

threshold_results = {}

for name in MODELS:
    pipe = make_pipeline(name)
    pipe.fit(X_tr, y_tr)
    if hasattr(pipe.named_steps['clf'], 'predict_proba'):
        proba = pipe.predict_proba(X_val)[:, 1]
        best_t, best_f = _best_threshold(y_val.values, proba)
        default_f = f1_score(y_val, (proba >= 0.5).astype(int), average='macro')
        threshold_results[name] = {'threshold': best_t, 'val_f1_tuned': best_f, 'val_f1_default': default_f}
        print(f'{name:6s}  default={default_f:.4f}  tuned={best_f:.4f}  threshold={best_t:.2f}')
    else:
        preds = pipe.predict(X_val)
        f = f1_score(y_val, preds, average='macro')
        threshold_results[name] = {'threshold': 0.5, 'val_f1_tuned': f, 'val_f1_default': f}
        print(f'{name:6s}  f1={f:.4f} (no proba)')

pd.DataFrame(threshold_results).T

## 4. Threshold curve for best model

In [ ]:
# Pick the model with the highest tuned val f1
best_model_name = max(threshold_results, key=lambda k: threshold_results[k]['val_f1_tuned'])
print('Best model:', best_model_name)

best_pipe = make_pipeline(best_model_name)
best_pipe.fit(X_tr, y_tr)
proba_val = best_pipe.predict_proba(X_val)[:, 1]

thresholds = np.linspace(0.05, 0.95, 181)
f1s = [f1_score(y_val, (proba_val >= t).astype(int), average='macro', zero_division=0) for t in thresholds]

best_idx = np.argmax(f1s)
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(thresholds, f1s)
ax.axvline(thresholds[best_idx], color='red', linestyle='--',
           label=f'best t={thresholds[best_idx]:.2f}, f1={f1s[best_idx]:.4f}')
ax.axvline(0.5, color='grey', linestyle=':', label='default t=0.50')
ax.set_xlabel('Decision threshold')
ax.set_ylabel('Macro F1')
ax.set_title(f'{best_model_name} — threshold vs Macro F1')
ax.legend()
plt.tight_layout(); plt.show()

## 5. Feature importance (best model)

In [ ]:
# Refit on all training data
best_pipe.fit(X_train, y)
clf = best_pipe.named_steps['clf']

if hasattr(clf, 'feature_importances_'):
    prep = best_pipe.named_steps['prep']
    try:
        feat_names = prep.get_feature_names_out()
    except Exception:
        feat_names = [f'f{i}' for i in range(len(clf.feature_importances_))]

    imp = pd.Series(clf.feature_importances_, index=feat_names).sort_values(ascending=False)
    imp.head(20).plot(kind='barh', figsize=(8, 6), title=f'{best_model_name} — top 20 feature importances')
    plt.gca().invert_yaxis()
    plt.tight_layout(); plt.show()
else:
    print('This model does not expose feature_importances_')

## 6. Write best submission

In [ ]:
best_t = threshold_results[best_model_name]['threshold']
test_proba = best_pipe.predict_proba(X_test)[:, 1]
preds = (test_proba >= best_t).astype(int)

sub_dir = ROOT / 'submissions'
sub_dir.mkdir(parents=True, exist_ok=True)

submission = pd.DataFrame({ID_COL: test_ids, TARGET_COL: preds})
out_path = sub_dir / f'{best_model_name}_tuned.csv'
submission.to_csv(out_path, index=False)

print(f'Saved: {out_path}')
print(submission[TARGET_COL].value_counts())
submission.head()

## 7. (Optional) Optuna hyperparameter search

Run this cell only when you have time — it can take 30-60 min.
Change `N_TRIALS` up or down depending on your time budget.

In [ ]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

N_TRIALS = 30  # increase to 100 for a proper search

def objective(trial):
    from lightgbm import LGBMClassifier
    from sklearn.compose import ColumnTransformer
    from sklearn.impute import SimpleImputer
    from sklearn.preprocessing import OneHotEncoder

    params = {
        'n_estimators':   trial.suggest_int('n_estimators', 200, 1000),
        'learning_rate':  trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'num_leaves':     trial.suggest_int('num_leaves', 15, 127),
        'subsample':      trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha':      trial.suggest_float('reg_alpha', 1e-4, 10.0, log=True),
        'reg_lambda':     trial.suggest_float('reg_lambda', 1e-4, 10.0, log=True),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
        'is_unbalance': True,
        'n_jobs': -1,
        'random_state': 42,
        'verbose': -1,
    }
    clf = LGBMClassifier(**params)
    cat_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False, max_categories=50)),
    ])
    prep = ColumnTransformer([
        ('num', SimpleImputer(strategy='median'), num_cols),
        ('cat', cat_pipe, cat_cols),
    ])
    pipe = Pipeline([('prep', prep), ('clf', clf)])
    cv_scores = cross_val_score(pipe, X_train, y, cv=3, scoring='f1_macro', n_jobs=1)
    return cv_scores.mean()

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

print('Best value:', study.best_value)
print('Best params:', study.best_params)